# Full CNN Walkthrough for CIFAR-10

This notebook puts the full first CNN baseline in one place.

It includes:
- the CNN architecture
- the training setup
- the loss function choice
- the hyperparameters and where to change them
- simple explanations of what each part is doing


## 1. Imports and Project Path

The only external project dependency we import is the CIFAR-10 dataloader that we already built in `SRC/cifar10_data.py`.


In [ ]:
%matplotlib inline

from pathlib import Path
from datetime import datetime
import json
import sys

import matplotlib.pyplot as plt
import torch
from torch import nn

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from SRC.cifar10_data import CIFAR10_CLASSES, CIFAR10_MEAN, CIFAR10_STD, get_cifar10_dataloaders


## 2. Hyperparameters

This is the main place to change the training setup.

**Training hyperparameters** control how learning happens:
- `BATCH_SIZE`
- `EPOCHS`
- `LEARNING_RATE`
- `WEIGHT_DECAY`
- `USE_AUGMENTATION`
- `NUM_WORKERS`

**Architecture hyperparameters** control the model structure:
- `CONV_CHANNELS`
- `DROPOUT`
- `NUM_CLASSES`


In [ ]:
BATCH_SIZE = 256
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
USE_AUGMENTATION = False
NUM_WORKERS = 0
VALIDATION_SPLIT = 0.1
SEED = 42

CONV_CHANNELS = (32, 64, 128)
DROPOUT = 0.4
NUM_CLASSES = 10


## 3. Load the Data

We load train, validation, and test dataloaders. The images are already normalized using the CIFAR-10 mean and standard deviation from EDA.


In [ ]:
train_loader, val_loader, test_loader = get_cifar10_dataloaders(
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    augment=USE_AUGMENTATION,
    seed=SEED,
    num_workers=NUM_WORKERS,
)

images, labels = next(iter(train_loader))
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Batch image shape: {tuple(images.shape)}")
print(f"Batch label shape: {tuple(labels.shape)}")


## 4. Full CNN Code

This is the full first CNN baseline.

How to read it:
- `conv1_1` learns 32 filters over the raw RGB image
- `conv1_2` recombines those first feature maps
- `pool1` reduces spatial size from `32x32` to `16x16`
- the same pattern repeats at deeper levels
- `global_pool` keeps the strongest high-level information while discarding exact location
- the final linear layer maps the learned representation to 10 class scores


In [ ]:
class SimpleCIFAR10CNN(nn.Module):
    def __init__(self, conv_channels=(32, 64, 128), num_classes=10, dropout=0.4):
        super().__init__()

        c1, c2, c3 = conv_channels

        self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=c1, kernel_size=3, padding=1)
        self.relu1_1 = nn.ReLU()
        self.conv1_2 = nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, padding=1)
        self.relu1_2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2_1 = nn.Conv2d(in_channels=c1, out_channels=c2, kernel_size=3, padding=1)
        self.relu2_1 = nn.ReLU()
        self.conv2_2 = nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, padding=1)
        self.relu2_2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3_1 = nn.Conv2d(in_channels=c2, out_channels=c3, kernel_size=3, padding=1)
        self.relu3_1 = nn.ReLU()
        self.conv3_2 = nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, padding=1)
        self.relu3_2 = nn.ReLU()
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(in_features=c3, out_features=num_classes),
        )

    def forward(self, x):
        x = self.conv1_1(x)
        x = self.relu1_1(x)
        x = self.conv1_2(x)
        x = self.relu1_2(x)
        x = self.pool1(x)

        x = self.conv2_1(x)
        x = self.relu2_1(x)
        x = self.conv2_2(x)
        x = self.relu2_2(x)
        x = self.pool2(x)

        x = self.conv3_1(x)
        x = self.relu3_1(x)
        x = self.conv3_2(x)
        x = self.relu3_2(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

    @torch.no_grad()
    def describe_feature_shapes(self, input_shape=(1, 3, 32, 32)):
        x = torch.zeros(input_shape)
        shapes = {"input": tuple(x.shape)}
        x = self.pool1(self.relu1_2(self.conv1_2(self.relu1_1(self.conv1_1(x)))))
        shapes["after_block1"] = tuple(x.shape)
        x = self.pool2(self.relu2_2(self.conv2_2(self.relu2_1(self.conv2_1(x)))))
        shapes["after_block2"] = tuple(x.shape)
        x = self.global_pool(self.relu3_2(self.conv3_2(self.relu3_1(self.conv3_1(x)))))
        shapes["after_block3"] = tuple(x.shape)
        x = self.classifier(x)
        shapes["logits"] = tuple(x.shape)
        return shapes


In [ ]:
model = SimpleCIFAR10CNN(conv_channels=CONV_CHANNELS, num_classes=NUM_CLASSES, dropout=DROPOUT)
print(model)
model.describe_feature_shapes()


## 5. Loss Choice

We use `CrossEntropyLoss` because CIFAR-10 is a **single-label multi-class classification** problem.

That means:
- each image belongs to exactly one of 10 classes
- the model outputs 10 raw class scores, called logits
- the loss should encourage the correct class score to be high and the incorrect ones to be lower

Cross-entropy is the standard choice because it strongly penalizes confident mistakes and works directly with integer class labels.


In [ ]:
criterion = nn.CrossEntropyLoss()
criterion


## 6. Training Utilities

These are the core training functions.

- `train_one_epoch` does forward pass, loss computation, backpropagation, and optimizer updates.
- `evaluate` measures loss and accuracy without changing the weights.


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_examples += batch_size

    return {
        "loss": running_loss / running_examples,
        "accuracy": running_correct / running_examples,
    }


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        logits = model(inputs)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_examples += batch_size

    return {
        "loss": running_loss / running_examples,
        "accuracy": running_correct / running_examples,
    }


## 7. Train the CNN

This is the full training loop. The notebook stores training history so we can plot the learning curves and later save a checkpoint.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

model = SimpleCIFAR10CNN(conv_channels=CONV_CHANNELS, num_classes=NUM_CLASSES, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
best_val_accuracy = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    record = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    }
    history.append(record)

    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} | train_acc={train_metrics['accuracy']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['accuracy']:.4f}"
    )


## 8. Plot the Learning Curves

These two plots tell us whether the model is actually learning and whether training and validation are moving in a healthy direction.


In [ ]:
epoch_numbers = [entry["epoch"] for entry in history]
train_loss = [entry["train_loss"] for entry in history]
val_loss = [entry["val_loss"] for entry in history]
train_acc = [entry["train_accuracy"] for entry in history]
val_acc = [entry["val_accuracy"] for entry in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epoch_numbers, train_loss, marker="o", label="Train loss")
axes[0].plot(epoch_numbers, val_loss, marker="o", label="Validation loss")
axes[0].set_title("CNN Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(epoch_numbers, train_acc, marker="o", label="Train accuracy")
axes[1].plot(epoch_numbers, val_acc, marker="o", label="Validation accuracy")
axes[1].set_title("CNN Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Evaluate on the Test Set

We reload the best validation model before testing, so the final test score reflects the strongest checkpoint from training.


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(model, test_loader, criterion, device)

print(f"Best validation accuracy: {best_val_accuracy:.4f}")
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")


## 10. Save the Model and Summary

This cell saves the trained checkpoint and a JSON summary under the `Data` folder.


In [ ]:
run_dir = PROJECT_ROOT / "Data" / "cnn_notebook_full_runs" / datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = run_dir / "best_cnn_notebook.pt"
torch.save(model.state_dict(), checkpoint_path)

summary = {
    "model": "SimpleCIFAR10CNN",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "validation_split": VALIDATION_SPLIT,
    "augment": USE_AUGMENTATION,
    "conv_channels": CONV_CHANNELS,
    "best_val_accuracy": best_val_accuracy,
    "test_loss": test_metrics["loss"],
    "test_accuracy": test_metrics["accuracy"],
    "history": history,
}

summary_path = run_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Saved checkpoint to: {checkpoint_path}")
print(f"Saved summary to: {summary_path}")


## 11. Inspect a Few Feature Maps

This helps answer the question: *what is the CNN actually learning?*

Early feature maps often respond to simple structure such as edges, color changes, or local texture. Deeper maps respond to richer combinations of those earlier patterns.


In [ ]:
def denormalize(image_tensor):
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    return (image_tensor.cpu() * std + mean).clamp(0.0, 1.0)

sample_image, sample_label = test_loader.dataset[0]
sample_batch = sample_image.unsqueeze(0).to(device)

with torch.no_grad():
    x = model.conv1_1(sample_batch)
    conv1_maps = x.detach().cpu()
    logits = model(sample_batch).detach().cpu()

predicted_label = int(logits.argmax(dim=1).item())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, axis in enumerate(axes.flat):
    axis.imshow(conv1_maps[0, index], cmap="viridis")
    axis.set_title(f"conv1_1 map {index}")
    axis.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(denormalize(sample_image).permute(1, 2, 0))
plt.title(f"True: {CIFAR10_CLASSES[sample_label]} | Predicted: {CIFAR10_CLASSES[predicted_label]}")
plt.axis("off")
plt.show()


## 12. What to Tune Next

The next hyperparameters to tune are:
- `USE_AUGMENTATION`
- `EPOCHS`
- `LEARNING_RATE`
- `WEIGHT_DECAY`
- `DROPOUT`
- `CONV_CHANNELS`

A good next experiment would be to keep the architecture the same but turn on augmentation, then compare the result against this no-augmentation baseline.
